# 1 — Exploratory Data Analysis: GTZAN Genre Dataset

| Purpose | File |
|---|---|
| EDA & visualization | `features_30_sec.csv` — 1 row per 30-second song (1,000 rows) |
| Model training | `features_3_sec.csv` — 10 × 3-second clips per song (9,990 rows) |

**Sections:**
1. Load & Inspect Data
2. Class Distribution
3. Waveform Plots
4. Spectrogram Plots
5. MFCC Plots
6. Audio Duration Distribution
   - 6.1 Duration Outlier Detail
7. Sample Rate Check
8. Corrupted / Missing File Check
9. Written Summary

---
## Imports & Configuration

Standard scientific stack plus `librosa` for audio loading and display. `EXPECTED_GENRES` defines the 10 GTZAN classes used throughout for validation and plotting. `GENRES` is the sorted list used to keep subplot order consistent.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from IPython.display import Image, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

EXPECTED_GENRES = {
    'blues', 'classical', 'country', 'disco', 'hiphop',
    'jazz', 'metal', 'pop', 'reggae', 'rock',
}
GENRES = sorted(EXPECTED_GENRES)

---
## Helper Functions

Reusable utilities that keep the EDA cells below concise. `find_repo_root` locates the project root so paths work regardless of where the notebook is opened from. The audio helpers (`discover_audio_files`, `build_genre_counts`, `validate_gtzan_layout`) scan the raw `.wav` files on disk; the plot helpers write PNGs to `docs/images/`.

In [ ]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    candidates = []
    for path in (start, *start.parents):
        candidates.append(path)
        candidates.append(path / 'audio-genre-classifier')
    for candidate in candidates:
        if (candidate / 'docs' / 'requirements.txt').exists() and (candidate / 'code').exists():
            return candidate
    raise FileNotFoundError('Could not find the audio-genre-classifier repo root.')

def discover_audio_files(dataset_path: Path) -> pd.DataFrame:
    dataset_path = Path(dataset_path)
    if not dataset_path.exists():
        raise FileNotFoundError(
            f'Dataset path not found: {dataset_path}. '
            'Place GTZAN at data/genres_original before running.'
        )
    rows = []
    for genre_dir in sorted(p for p in dataset_path.iterdir() if p.is_dir()):
        for wav_file in sorted(genre_dir.glob('*.wav')):
            rows.append({
                'genre': genre_dir.name,
                'file_name': wav_file.name,
                'file_path': str(wav_file),
            })
    return pd.DataFrame(rows, columns=['genre', 'file_name', 'file_path'])


def build_genre_counts(dataset_path: Path) -> pd.DataFrame:
    audio_files = discover_audio_files(dataset_path)
    return (
        audio_files.groupby('genre', as_index=False)
        .size()
        .rename(columns={'size': 'count'})
        .sort_values('genre')
        .reset_index(drop=True)
    )


def validate_gtzan_layout(genre_counts: pd.DataFrame) -> pd.DataFrame:
    counts_by_genre = dict(zip(genre_counts['genre'], genre_counts['count']))
    rows = []
    for genre in sorted(EXPECTED_GENRES | set(counts_by_genre)):
        count = counts_by_genre.get(genre, 0)
        if genre not in EXPECTED_GENRES:
            status = 'unexpected genre'
        elif count == 100:
            status = 'ok'
        elif count == 0:
            status = 'missing genre'
        else:
            status = 'expected 100 files'
        rows.append({'genre': genre, 'count': count, 'status': status})
    return pd.DataFrame(rows, columns=['genre', 'count', 'status'])


def plot_genre_distribution(genre_counts: pd.DataFrame, output_path: Path) -> None:
    ax = genre_counts.plot(
        kind='bar', x='genre', y='count',
        legend=False, color='steelblue', figsize=(10, 5),
    )
    ax.set_title('GTZAN Genre Class Distribution')
    ax.set_xlabel('Genre')
    ax.set_ylabel('Number of .wav files')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150)
    plt.close('all')


def build_audio_metadata(dataset_path: Path, limit=None) -> pd.DataFrame:
    audio_files = discover_audio_files(dataset_path)
    if limit is not None:
        audio_files = audio_files.head(limit)
    rows = []
    for row in audio_files.itertuples(index=False):
        samples, sample_rate = librosa.load(row.file_path, sr=None, mono=True)
        rows.append({
            'genre': row.genre,
            'file_name': row.file_name,
            'sample_rate': sample_rate,
            'sample_count': len(samples),
            'duration_seconds': librosa.get_duration(y=samples, sr=sample_rate),
        })
    return pd.DataFrame(rows)


def build_example_file_paths(dataset_path: Path) -> dict:
    audio_files = discover_audio_files(dataset_path)
    examples = {}
    for genre in sorted(EXPECTED_GENRES):
        genre_files = audio_files[audio_files['genre'] == genre].sort_values('file_name')
        if not genre_files.empty:
            examples[genre] = Path(genre_files.iloc[0]['file_path'])
    return examples


def plot_mfcc_examples(
    dataset_path: Path,
    output_path: Path,
    n_mfcc: int = 13,
    duration: float = 30.0,
) -> None:
    examples = build_example_file_paths(dataset_path)
    fig, axes = plt.subplots(5, 2, figsize=(12, 14), constrained_layout=True)
    axes = axes.flatten()
    image = None
    for ax, genre in zip(axes, sorted(examples)):
        samples, sample_rate = librosa.load(
            str(examples[genre]), sr=None, mono=True, duration=duration,
        )
        mfccs = librosa.feature.mfcc(y=samples, sr=sample_rate, n_mfcc=n_mfcc)
        image = librosa.display.specshow(mfccs, x_axis='time', ax=ax, cmap='magma')
        ax.set_title(f'{genre}: {examples[genre].name}')
        ax.set_ylabel('MFCC')
        ax.set_xlabel('Time')
    for ax in axes[len(examples):]:
        ax.axis('off')
    fig.suptitle('GTZAN MFCC Examples by Genre', fontsize=16)
    if image is not None:
        fig.colorbar(image, ax=axes, format='%+2.0f dB', shrink=0.65)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


print('Helper functions defined.')

---
## 1. Load & Inspect Data

In [ ]:
repo_root = find_repo_root(Path.cwd())
dataset_path = repo_root / 'data' / 'genres_original'
images_dir   = repo_root / 'docs' / 'images'

df = pd.read_csv(repo_root / 'data' / 'features_30_sec.csv')
print(f'Shape: {df.shape}  (1 row per 30-second song)')

cols_preview = list(df.columns)
print(f'Columns: {cols_preview} ... [{len(df.columns)} total]')
df.head()

In [ ]:
df.describe()

Key things to notice in the table above:

- **Songs aren't all exactly the same length** — the `length` column (how many audio samples are in each file) goes from 660,000 up to 675,808. Most songs are almost identical, but a few are very slightly shorter or longer than 30 seconds. We dig into this in Section 6.
- **The numbers are on very different scales** — for example, `spectral_centroid_mean` can be in the thousands (it measures frequency in Hz), while `chroma_stft_mean` is always between 0 and 1. If we fed these raw numbers into a model, the big-number features would dominate unfairly. We'll rescale everything to the same range in the next notebook — **standardization**.
- **Some columns have extreme high values** — for several `*_var` columns, the max is way higher than the 75th percentile. For example, `spectral_bandwidth_var` has a max of 694,784 but 75% of songs are below 182,371. This means a small number of clips sound very unusual compared to the rest.
- **MFCC columns follow a pattern** — the `*_mean` columns hover near 0 (some positive, some negative), while the `*_var` columns are always positive. This is expected: MFCCs are designed to be centered, and variance is always non-negative.

In [ ]:
print('Data types:')
print(df.dtypes.value_counts())
print()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicated values: {df.duplicated().sum().sum()}')


Quick data quality check — confirms column types, missing values, and duplicate rows. All 57 audio features are `float64`, `filename` and `label` are strings, and `length` is `int64`. Zero nulls and zero duplicates means the CSV is clean and ready to use.

---
## 2. Class Distribution

GTZAN should have 100 `.wav` files per genre. The QA table flags any unexpected counts.

In [ ]:
genre_counts = build_genre_counts(dataset_path)
print('File counts per genre:')
display(genre_counts)
print()
layout_validation = validate_gtzan_layout(genre_counts)
print('Layout validation:')
display(layout_validation)

In [ ]:
genre_chart_path = images_dir / 'genre_distribution.png'
plot_genre_distribution(genre_counts, output_path=genre_chart_path)
display(Image(filename=str(genre_chart_path)))
print(f'Saved to {genre_chart_path}')

All 10 bars are the same height. The dataset is perfectly balanced with exactly 100 songs per genre, so no class will be over or under-represented during training.

---
## 3. Waveform Plots

Raw amplitude over time for one representative 30-second clip per genre. Reveals differences in energy, dynamics, and rhythmic density across genres.

In [ ]:
examples = build_example_file_paths(dataset_path)

fig, axes = plt.subplots(5, 2, figsize=(14, 18))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    try:
        y, sr = librosa.load(str(examples[genre]), sr=None, duration=30.0)
        librosa.display.waveshow(y, sr=sr, ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].set_title(genre.capitalize(), fontsize=12)
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Amplitude')
    except Exception as e:
        axes[i].set_title(f'{genre.capitalize()} - ERROR')
        axes[i].text(0.5, 0.5, str(e), ha='center', va='center', transform=axes[i].transAxes)

fig.suptitle('Waveforms - One 30-Second Clip per Genre', fontsize=16, y=1.01)
plt.tight_layout()
waveform_path = images_dir / 'waveforms.png'
plt.savefig(waveform_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {waveform_path}')

The height of the wave shows loudness at each moment in time. Notice how classical has quiet and loud passages while metal stays dense and loud throughout. These amplitude patterns are one of the things the model will learn to distinguish.

---
## 4. Spectrogram Plots

A mel spectrogram converts audio into a 2-D frequency x time image using a perceptually scaled frequency axis. Primary input for audio CNNs: shows tonal complexity, harmonic content, and rhythmic texture.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    try:
        y, sr = librosa.load(str(examples[genre]), sr=None, duration=30.0)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(
            S_dB, x_axis='time', y_axis='mel', sr=sr, fmax=8000, ax=axes[i], cmap='magma'
        )
        fig.colorbar(img, ax=axes[i], format='%+2.0f dB')
        axes[i].set_title(genre.capitalize(), fontsize=12)
    except Exception as e:
        axes[i].set_title(f'{genre.capitalize()} - ERROR')
        axes[i].text(0.5, 0.5, str(e), ha='center', va='center', transform=axes[i].transAxes)

fig.suptitle('Mel Spectrograms - One 30-Second Clip per Genre', fontsize=16, y=1.01)
plt.tight_layout()
spec_path = images_dir / 'spectrograms.png'
plt.savefig(spec_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {spec_path}')

Brighter colors mean more energy at that frequency and time. Classical shows concentrated bright bands at specific pitches, while metal and rock spread bright color across a wide frequency range, these distinct shapes are why spectrograms work well as input to image-based models.

---
## 5. MFCC Plots

MFCCs compress the spectrogram into the most perceptually meaningful features. The first 13 coefficients over time show what the model will learn from. The same features are captured as mean/variance columns in the CSV.

In [ ]:
mfcc_chart_path = images_dir / 'mfcc_examples_by_genre.png'
plot_mfcc_examples(dataset_path=dataset_path, output_path=mfcc_chart_path)
display(Image(filename=str(mfcc_chart_path)))
print(f'Saved to {mfcc_chart_path}')

Each row is one MFCC and each column is a point in time, color shows the value. Classical has smooth, slowly-changing patterns while metal and rock show rapid, scattered color changes, reflecting their noisier harmonic structure.

---
## 6. Audio Duration Distribution

The `length` column in the CSV is in samples. We convert to seconds using the native sample rate and check for inconsistencies across songs and genres.

In [ ]:
sample_file = str(dataset_path / 'blues' / 'blues.00000.wav')
_, native_sr = librosa.load(sample_file, sr=None, duration=1.0)
print(f'Native sample rate: {native_sr} Hz')

df['duration_sec'] = df['length'] / native_sr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mean_dur = df['duration_sec'].mean()
axes[0].hist(df['duration_sec'], bins=30, color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Clip Durations (all songs)', fontsize=13)
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].axvline(mean_dur, color='red', linestyle='--', label=f'Mean: {mean_dur:.2f}s')
axes[0].legend()

duration_by_genre = [df[df['label'] == g]['duration_sec'].values for g in GENRES]
axes[1].boxplot(duration_by_genre, labels=GENRES, vert=True)
axes[1].set_title('Duration per Genre', fontsize=13)
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Duration (seconds)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
dur_path = images_dir / 'duration_distribution.png'
plt.savefig(dur_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {dur_path}')
print()
print('Duration stats (seconds):')
print(df['duration_sec'].describe().round(3))
n_unique = df['duration_sec'].nunique()
print(f'Unique duration values: {n_unique}')

Almost all songs cluster tightly near 30 seconds, and the box plots show this is consistent across every genre. The small spread confirms duration alone won't help the model tell genres apart.

### 6.1 Duration Outlier Detail

The histogram above shows most songs cluster near 30.013 s (661,794 samples), but a small tail exists on both ends. This subsection identifies the exact files affected and traces the downstream impact on `features_3_sec.csv`.

In [ ]:
CLIP_SR = 22050  # confirmed native sample rate

# Unique length values across all 1,000 songs
length_dist = (
    df.groupby('length')
    .agg(file_count=('filename', 'count'))
    .reset_index()
    .assign(duration_sec=lambda d: (d['length'] / CLIP_SR).round(3))
    .sort_values('length')
    [['length', 'duration_sec', 'file_count']]
    .reset_index(drop=True)
)
print('Unique sample-count values in features_30_sec.csv:')
display(length_dist)

# Cross-reference with features_3_sec.csv to count clips per parent song
df3 = pd.read_csv(CSV_TRAIN)
df3['parent_song'] = df3['filename'].str.rsplit('.', n=2).str[0]
df['parent_song']  = df['filename'].str.rsplit('.', n=1).str[0]

clips_per_song = df3.groupby('parent_song').size().rename('clip_count')

print(f'\nTotal rows in features_3_sec.csv : {len(df3)}  (expected 10,000 = 1,000 songs × 10 clips)')
print(f'Songs with fewer than 10 clips   : {(clips_per_song < 10).sum()}')

nine_clip_detail = (
    clips_per_song[clips_per_song < 10]
    .reset_index()
    .merge(df[['parent_song', 'label', 'length', 'duration_sec']], on='parent_song', how='left')
    [['parent_song', 'label', 'length', 'duration_sec', 'clip_count']]
    .sort_values('label')
    .reset_index(drop=True)
)
print()
print('Songs that produced only 9 three-second clips:')
display(nine_clip_detail)

---
## 7. Sample Rate Check

Loading every `.wav` file briefly (0.1s) to confirm they all share the same sample rate. A mismatch causes problems downstream in preprocessing.

In [ ]:
print('Running sample rate check across all .wav files...')
sr_results = {}
sr_errors  = []

for genre in GENRES:
    genre_dir = dataset_path / genre
    for wav_file in sorted(genre_dir.glob('*.wav')):
        try:
            _, sr = librosa.load(str(wav_file), sr=None, duration=0.1)
            sr_results[wav_file.name] = sr
        except Exception as e:
            sr_errors.append((wav_file.name, str(e)))

sr_series = pd.Series(list(sr_results.values()))
print('Sample rate value counts:')
print(sr_series.value_counts().to_string())
print()
print(f'Total files checked : {len(sr_results)}')
print(f'Errors during check : {len(sr_errors)}')

if sr_series.nunique() == 1:
    sr_val = int(sr_series.iloc[0])
    print()
    print(f'All files share sample rate: {sr_val} Hz')
else:
    print()
    print('WARNING: Mixed sample rates - resampling required in preprocessing.')

---
## 8. Corrupted / Missing File Check

Full load of every `.wav` file to surface files that fail to load, are silent (near-zero energy), or have unexpected durations.

In [ ]:
load_errors   = []
silent_files  = []
anomalous_dur = []
all_durations = []

EXPECTED_DUR  = 30.0
DUR_TOLERANCE = 2.0

print('Running full integrity check (this may take a minute)...')

for genre in GENRES:
    genre_dir = dataset_path / genre
    for wav_file in sorted(genre_dir.glob('*.wav')):
        try:
            y, sr = librosa.load(str(wav_file), sr=None)
            dur = len(y) / sr
            all_durations.append(dur)
            rms = float(np.sqrt(np.mean(y ** 2)))
            if rms < 1e-4:
                silent_files.append((wav_file.name, round(rms, 6)))
            if abs(dur - EXPECTED_DUR) > DUR_TOLERANCE:
                anomalous_dur.append((wav_file.name, round(dur, 2)))
        except Exception as e:
            load_errors.append((wav_file.name, str(e)))

n_total = len(all_durations) + len(load_errors)
print(f'Total .wav files scanned  : {n_total}')
print(f'Successfully loaded       : {len(all_durations)}')
print(f'Load errors               : {len(load_errors)}')
print(f'Silent files (RMS < 1e-4) : {len(silent_files)}')
print(f'Anomalous duration files  : {len(anomalous_dur)}')

if load_errors:
    print()
    print('Load errors:')
    for f, e in load_errors:
        print(f'  {f}: {e}')
if silent_files:
    print()
    print('Silent files:')
    for f, r in silent_files:
        print(f'  {f}: RMS={r}')
if anomalous_dur:
    print()
    print('Anomalous durations:')
    for f, d in anomalous_dur:
        print(f'  {f}: {d}s')
if not load_errors and not silent_files and not anomalous_dur:
    print()
    print('All files passed integrity checks.')

---
## 9. Summary

**Class Distribution**
The dataset is evenly balanced — 100 songs per genre, 1,000 songs total in `features_30_sec.csv`. The QA validation table confirmed all 10 expected genres are present with the correct file count. No class weighting will be needed during training.

**Waveforms**
Clear visual differences appear across genres. Classical clips show wide dynamic range with quiet and loud passages. Metal and rock are dense and high-amplitude throughout (brickwall compression). Reggae and hiphop show rhythmic gaps between hits. Blues and jazz have smoother, more continuous waveforms.

**Spectrograms**
Mel spectrograms reveal distinct harmonic and noise signatures per genre. Classical concentrates energy in lower-mid frequencies with clear overtones. Metal and rock spread energy broadly across high frequencies. Hiphop and reggae show strong low-frequency (bass) dominance. These visual differences suggest mel spectrograms alone could serve as CNN input.

**MFCCs**
MFCC patterns differ noticeably across genres. Classical shows slowly varying, structured coefficients; metal and rock show high-variance, noisy patterns. The first few coefficients (MFCC 1–4) carry the most energy and will dominate model learning. The mean/variance summaries in the CSV are reasonable representations of these patterns.

**Audio Duration**
Source `.wav` files are approximately 30 seconds but not exactly uniform. The `length` column contains 35 distinct sample-count values ranging from 660,000 (29.932 s) to 675,808 (30.649 s); the majority (640 songs) are 661,794 samples (30.013 s). The integrity check in Section 8 used a ±2-second tolerance, so none of these files were flagged — but the variation has a real downstream effect: **10 songs across classical, country, disco, hiphop, and rock are marginally short and produced only 9 three-second clips in `features_3_sec.csv` instead of 10**, making the dataset 9,990 rows rather than 10,000. No resampling is needed, but the train/val/test split must be done at the parent-song level (see Preprocessing Notes below).

**Sample Rate**
All successfully loaded files share a native sample rate of 22,050 Hz (GTZAN standard). No resampling required.

**Corrupted / Missing Files**
999 of 1,000 files loaded successfully. `jazz.00054.wav` fails to load entirely — its raw audio file is corrupt. The row is present in `features_30_sec.csv` because the CSV was pre-computed, but the file cannot be used and **must be excluded in `2_transform.ipynb`**.

### Preprocessing Notes for `2_transform.ipynb`

- Use **`features_3_sec.csv`** for model training (9,990 rows across 10 genres — not 10,000)
- Drop `filename` and `length` — metadata, not features
- Encode `label` to integers
- Standardize features with `StandardScaler` fit on the train set only, then apply to val and test
- **Split on parent song, not on individual clips** — each song's 9–10 clips share the same source audio; a clip-level split leaks the same recording into both train and test
- Drop `parent_song == 'jazz.00054'` rows before splitting (corrupt source file)